# 0DTE Heston Calibration Notebook

This notebook calibrates Heston-style models to one true Monday-Friday week of SPX 0DTE option quotes. At each quote time it keeps the closest 10 strikes below the underlying and the closest 10 strikes above the underlying. The quoted option data contain bid, ask, mid, strike, underlying spot, quote time, and same-day expiration.

The notebook is self-contained. All model definitions, calibration targets, variables, diagnostics, and plotting conventions are defined here.

## Data Scope

For each quote time $t$:

- $S_t$: underlying index level at the quote time.
- $K_i$: strike of option $i$.
- $B_i$: bid quote of option $i$.
- $A_i$: ask quote of option $i$.
- $M_i = (B_i + A_i)/2$: mid quote.
- $\tau_t = (T-t)/\mathrm{year}$: time to same-day expiry in years.
- $r$: risk-free rate. In this notebook, the 0DTE baseline uses `RISK_FREE_RATE = 0`.
- $N$: number of strikes in a cross-section, normally $20$.

Filtering:

- same-day expiration only,
- calls only in the raw data,
- positive and finite spot, strike, bid/ask/mid,
- relative spread below `MAX_REL_SPREAD`,
- quote times with at least `CALIBRATION_MIN_TAU_MINUTES` remaining.

## Heston Stochastic Volatility Model

The risk-neutral Heston model is

$$dS_t = rS_t\,dt + \sqrt{v_t}S_t\,dW_{1,t}^{\mathbb{Q}}$$

$$dv_t = \kappa(\theta-v_t)\,dt + \sigma\sqrt{v_t}\,dW_{2,t}^{\mathbb{Q}}$$

$$dW_{1,t}^{\mathbb{Q}}dW_{2,t}^{\mathbb{Q}}=\rho\,dt$$

The calibrated parameter vector is

$$\Theta_H=(v_0,\kappa,\theta,\sigma,\rho)$$

where:

- $v_0$: instantaneous variance at the quote time.
- $\kappa$: variance mean-reversion speed.
- $\theta$: long-run variance level.
- $\sigma$: volatility of variance, often called vol-of-vol.
- $\rho$: correlation between spot and variance shocks.
- $r$: risk-free rate.
- $\tau$: time to expiration.

The variance process is a CIR process. The soft Feller stability condition is

$$2\kappa\theta \ge \sigma^2$$

The notebook does not hard-reject violations, but adds a soft penalty:

$$\lambda_F\max(\sigma^2-2\kappa\theta,0)^2$$

## Characteristic Function

All deterministic Heston pricers use the log-price characteristic function

$$\phi(u)=\mathbb{E}^{\mathbb{Q}}\left[e^{iu\log S_T}\right]$$

Define

$$d(u)=\sqrt{(\rho\sigma iu-\kappa)^2+\sigma^2(iu+u^2)}$$

$$g(u)=\frac{\kappa-\rho\sigma iu-d(u)}{\kappa-\rho\sigma iu+d(u)}$$

$$C(u)=iu(\log S_0+r\tau)+\frac{\kappa\theta}{\sigma^2}\left[(\kappa-\rho\sigma iu-d(u))\tau-2\log\left(\frac{1-g(u)e^{-d(u)\tau}}{1-g(u)}\right)\right]$$

$$D(u)=\frac{\kappa-\rho\sigma iu-d(u)}{\sigma^2}\frac{1-e^{-d(u)\tau}}{1-g(u)e^{-d(u)\tau}}$$

Then

$$\phi(u)=e^{C(u)+D(u)v_0}$$

## 0DTE Calibration Target

Raw call prices are unstable near expiry because deep ITM calls are dominated by intrinsic value. A few cents of quote noise or spot timestamp mismatch can look like a large model error. The notebook therefore fits the option time value / OTM component rather than raw call price.

For calls, intrinsic value is

$$I_i=\max(S_t-K_ie^{-r\tau_t},0)$$

Market time value:

$$TV_i^{\mathrm{mkt}}=\max(M_i-I_i,0)$$

Model time value:

$$TV_i^{\mathrm{model}}(\Theta_H)=\max(P_i(\Theta_H)-I_i,0)$$

For strikes below spot, this behaves like a synthetic OTM put target by removing the deterministic forward/intrinsic part of the call. For strikes above spot, it is the OTM call value.

## Bid/Ask-Band Robust Loss

The calibration target is not distance to mid when the model is already inside the quoted market. The transformed bid and ask targets are

$$B_i^{\mathrm{TV}}=\max(B_i-I_i,0),\qquad A_i^{\mathrm{TV}}=\max(A_i-I_i,0)$$

The quote uncertainty scale is

$$s_i=\max\left(\frac{A_i-B_i}{2},\epsilon_{\mathrm{tick}},\epsilon_{\mathrm{rel}}\,TV_i^{\mathrm{mkt}}\right)$$

The bid/ask-band residual is

$$z_i(\Theta_H)=\frac{\max(TV_i^{\mathrm{model}}(\Theta_H)-A_i^{\mathrm{TV}},0)-\max(B_i^{\mathrm{TV}}-TV_i^{\mathrm{model}}(\Theta_H),0)}{s_i}$$

If model time value lies inside the transformed bid/ask interval, $z_i=0$.

The robust Huber loss is

$$
H_\delta(z)=
\begin{cases}
\frac{1}{2}z^2, & |z|\le \delta \\
\delta(|z|-\frac{1}{2}\delta), & |z|>\delta
\end{cases}
$$

The full objective is

$$\min_{\Theta_H}\frac{1}{N}\sum_{i=1}^{N}H_\delta(z_i(\Theta_H))+\lambda_T R(\Theta_H,\Theta_{H,\mathrm{prev}})+\lambda_F\max(\sigma^2-2\kappa\theta,0)^2$$

where:

- $R(\Theta_H,\Theta_{H,\mathrm{prev}})$: temporal regularization against the previous quote-time parameters.
- $\lambda_T$: temporal regularization weight.
- $\lambda_F$: Feller penalty weight.

## Full and Reduced Heston Modes

0DTE has only one ultra-short maturity at each quote time. In that setting, the full five-parameter Heston model is weakly identified: $v_0$, $\sigma$, and $\rho$ drive most of the short-time smile, while $\kappa$ and $\theta$ are much less identifiable.

The notebook therefore has two modes:

### Full Mode

Used when

$$\tau_{\mathrm{minutes}}\ge T_{\mathrm{reduced}}$$

and fits

$$v_0,\kappa,\theta,\sigma,\rho$$

### Reduced 0DTE Mode

Used when

$$\tau_{\mathrm{minutes}}< T_{\mathrm{reduced}}$$

and fits only

$$v_0,\sigma,\rho$$

while freezing

$$\kappa,\theta$$

to the previous quote-time values when available, otherwise to `FIXED_KAPPA_THETA`.

## Pricing / Compiler Methods

### 1. Carr-Madan Fourier Inversion

For log strike $k=\log K$ and damping $\alpha>0$:

$$C(K)=\frac{e^{-\alpha k}}{\pi}\int_0^\infty \operatorname{Re}\left(e^{-iuk}\frac{e^{-r\tau}\phi(u-i(\alpha+1))}{\alpha^2+\alpha-u^2+i(2\alpha+1)u}\right)du$$

The implementation uses an adaptive frequency grid:

- larger $u_{\max}$ when $\tau$ is small,
- larger $N$ when $u_{\max}$ increases,
- a maximum cap to avoid uncontrolled numerical cost.

### 2. COS Expansion

The COS method approximates option value by expanding the risk-neutral density on a finite interval $[a,b]$:

$$
C(K)\approx e^{-r\tau}\sum_{n=0}^{N-1}{}^{\prime}
\operatorname{Re}\left(
\phi\left(\frac{n\pi}{b-a}\right)e^{-in\pi a/(b-a)}
\right)V_n
$$

where:

- $V_n$: Fourier-cosine coefficient of the call payoff.
- The prime means the first term receives half weight.
- $[a,b]$: truncation interval used for the log-moneyness domain.

### 3. Broadie-Kaya-Style Monte Carlo Diagnostic

The notebook also fits a Broadie-Kaya-style Monte Carlo method as its own diagnostic compiler. It samples the CIR variance endpoint from a noncentral chi-square law:

$$v_T=c\chi^{\prime 2}_{d}(\lambda)$$

and approximates integrated variance by a trapezoidal bridge:

$$\int_t^T v_s\,ds \approx \frac{v_t+v_T}{2}\tau$$

Then

$$C(K)\approx e^{-r\tau}\frac{1}{N_{\mathrm{paths}}}\sum_{j=1}^{N_{\mathrm{paths}}}\max(S_T^{(j)}-K,0)$$

Because Monte Carlo has sampling noise, this method is interpreted as a diagnostic fit, not the primary production fit.

## Diagnostics

The notebook reports:

- raw RMSE to call mid,
- time-value RMSE,
- spread-normalized RMSE,
- bid/ask hit rate,
- total-variance RMSE,
- error by tau bucket,
- error by moneyness bucket,
- parameter paths,
- Carr-Madan vs COS pricer disagreement,
- bid/ask violation rows.

Total variance is

$$w=\sigma_{\mathrm{IV}}^2\tau$$

and total-variance error is

$$\Delta w=(\sigma_{\mathrm{IV,model}}^2-\sigma_{\mathrm{IV,mkt}}^2)\tau$$

This is often more stable than direct implied-vol error near expiry.

## 3D Plot Conventions

3D model-fit surfaces use:

- x-axis: strike $K$,
- y-axis: quote time,
- z-axis: selected metric such as mid, time value, fitted price, or error.

The high-contrast option-side sheets indicate calibration regions:

- blue/cyan sheet: synthetic put side, $K<S_t$,
- red/yellow sheet: call side, $K>S_t$,
- black sheet: underlying / ATM divider, $K=S_t$ through time.

These sheets are visual guides and do not change the calibration.

## References

- Heston: stochastic volatility model with CIR variance.
- Carr and Madan: Fourier inversion for option valuation.
- Fang and Oosterlee: COS pricing method.
- Broadie and Kaya: exact simulation framework for affine stochastic volatility.
- Andersen: efficient Heston Monte Carlo simulation.
- Forde, Jacquier, and Lee: small-time Heston smile behavior.
- Lord and Kahl: stable Fourier inversion considerations.


In [1]:
# Cell 1: imports, one-week 0DTE data scope, Heston model definition, and shared calibration utilities.
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from scipy.optimize import minimize, brentq, minimize_scalar, OptimizeResult
from scipy.stats import ncx2, norm

warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = Path(r"..\magisterka\2022_0dte_spy\2022").resolve()
OPTION_TYPE = "C"
N_0DTE_DAYS = 5
STRIKES_EACH_SIDE = 10
MIN_MID = 0.05
MAX_REL_SPREAD = 0.50
RISK_FREE_RATE = 0.0
NY_TZ = "America/New_York"
EXPIRY_CLOSE_HOUR = 16
YEAR_SECONDS = 365.25 * 24 * 60 * 60

CARR_MADAN_N = 256
CARR_MADAN_UMAX = 300.0
CARR_MADAN_ALPHA = 1.35
CARR_MADAN_UMAX_CAP = 600.0
CARR_MADAN_DU_TARGET = 0.50
CARR_MADAN_N_MAX = 2048
COS_N = 96
COS_L = 10.0
BK_CALIBRATION_PATHS = 768
BK_EVALUATION_PATHS = 3072
OPT_MAXITER_PER_START = 55
OPT_POLISH_MAXITER = 110
QUOTE_ERROR_FLOOR = 0.05
RELATIVE_ERROR_FLOOR = 0.02
HUBER_DELTA = 2.0
TEMPORAL_REGULARIZATION = 1e-3
CALIBRATION_PRICE_MODE = "time_value"  # "time_value" is more stable for 0DTE than raw call price.
CALIBRATION_LOSS_MODE = "bid_ask_band"  # zero residual while the model target is inside bid/ask target.
CALIBRATION_MIN_TAU_MINUTES = 30.0
USE_REDUCED_0DTE_HESTON = True
REDUCED_HESTON_TAU_MINUTES = 120.0
FIXED_KAPPA_THETA = (5.0, 0.04)
FELLER_PENALTY_WEIGHT = 1e-4
TOTAL_VARIANCE_IV_FLOOR = 0.05
TOTAL_VARIANCE_IV_CEIL = 8.0
RUN_MC_DIAGNOSTIC_BY_DEFAULT = True
PRICER_VALIDATION_MAX_SECTIONS = 5

PARAM_BOUNDS = [(1e-5, 4.0), (0.05, 80.0), (1e-5, 4.0), (0.01, 10.0), (-0.999, 0.999)]
PARAM_NAMES = ["v0", "kappa", "theta", "sigma", "rho"]
DEFAULT_START = np.array([0.04, 5.0, 0.04, 1.0, -0.5], dtype=float)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Cannot find SPX 0DTE data directory: {DATA_DIR}")


def file_date_from_name(file):
    match = re.search(r"(\d{4}-\d{2}-\d{2})", file.name)
    return pd.Timestamp(match.group(1)) if match else None


def file_has_true_0dte(file, file_date):
    probe = pd.read_parquet(file, columns=["expiration"])
    expiration_dates = set(pd.to_datetime(probe["expiration"]).dt.date.unique())
    return file_date.date() in expiration_dates


def select_first_full_0dte_week(data_dir):
    records = []
    for file in sorted(data_dir.glob("spx-options-*.parquet")):
        file_date = file_date_from_name(file)
        if file_date is None:
            continue
        records.append({"file": file, "date": file_date, "has_0dte": file_has_true_0dte(file, file_date)})

    meta = pd.DataFrame(records)
    if meta.empty:
        raise FileNotFoundError(f"No SPX option parquet files found in {data_dir}")

    iso = meta["date"].dt.isocalendar()
    meta["iso_year"] = iso["year"]
    meta["iso_week"] = iso["week"]
    meta["dow"] = meta["date"].dt.dayofweek

    for _, group in meta.groupby(["iso_year", "iso_week"], sort=True):
        true_days = group[group["has_0dte"]].sort_values("date")
        if {0, 1, 2, 3, 4}.issubset(set(true_days["dow"].tolist())):
            return true_days[true_days["dow"].isin([0, 1, 2, 3, 4])].head(N_0DTE_DAYS)["file"].tolist()

    raise FileNotFoundError(f"Could not find a full Monday-Friday true 0DTE week in {data_dir}")


def closest_strikes_around_spot(g, spot, each_side=10):
    g = g.sort_values("strike").drop_duplicates("strike").copy()
    below = g[g["strike"] < spot].tail(each_side)
    above = g[g["strike"] > spot].head(each_side)
    out = pd.concat([below, above]).drop_duplicates("strike").sort_values("strike")
    return out


def load_spx_0dte_week(files):
    cols = ["quote_datetime", "expiration", "strike", "option_type", "bid", "ask", "mid", "active_underlying_price", "moneyness"]
    frames = []
    for file in files:
        df = pd.read_parquet(file, columns=cols)
        df["quote_datetime"] = pd.to_datetime(df["quote_datetime"], utc=True).dt.tz_convert(NY_TZ)
        df["quote_date"] = df["quote_datetime"].dt.date
        df["expiration_date"] = pd.to_datetime(df["expiration"]).dt.date
        df = df[df["quote_date"] == df["expiration_date"]].copy()
        if df.empty:
            continue

        expiry_close_naive = pd.to_datetime(df["expiration_date"].astype(str) + f" {EXPIRY_CLOSE_HOUR:02d}:00:00")
        df["expiry_datetime"] = expiry_close_naive.dt.tz_localize(NY_TZ)
        df["tau"] = (df["expiry_datetime"] - df["quote_datetime"]).dt.total_seconds() / YEAR_SECONDS
        df = df[(df["tau"] > 0) & (df["option_type"].str.upper() == OPTION_TYPE)].copy()
        frames.append(df)

    if not frames:
        raise RuntimeError("No true 0DTE rows found in the selected one-week files.")

    out = pd.concat(frames, ignore_index=True)
    out["spread"] = out["ask"] - out["bid"]
    out["rel_spread"] = out["spread"] / out["mid"].clip(lower=1e-6)
    out = out[
        np.isfinite(out["mid"]) & np.isfinite(out["active_underlying_price"]) &
        (out["mid"] >= MIN_MID) & (out["active_underlying_price"] > 0) &
        (out["strike"] > 0) & (out["rel_spread"] <= MAX_REL_SPREAD)
    ].copy()
    return out


week_files = select_first_full_0dte_week(DATA_DIR)
raw_0dte = load_spx_0dte_week(week_files)

cross_sections = []
for qdt, g in raw_0dte.groupby("quote_datetime", sort=True):
    spot = float(g["active_underlying_price"].median())
    tau = float(g["tau"].median())
    tau_minutes = tau * YEAR_SECONDS / 60.0
    if tau_minutes < CALIBRATION_MIN_TAU_MINUTES:
        continue
    g = closest_strikes_around_spot(g, spot, STRIKES_EACH_SIDE)
    if len(g) < (2 * STRIKES_EACH_SIDE):
        continue
    cross_sections.append({
        "quote_datetime": qdt,
        "date": qdt.date().isoformat(),
        "time": qdt.strftime("%H:%M"),
        "minute_of_day": qdt.hour * 60 + qdt.minute,
        "spot": spot,
        "tau_years": tau,
        "tau_minutes": tau_minutes,
        "data": g[["strike", "mid", "bid", "ask", "moneyness"]].copy(),
    })

if not cross_sections:
    raise RuntimeError("No usable 0DTE cross-sections survived filtering.")

print(f"Data directory: {DATA_DIR}")
print("Using one full true 0DTE week:")
for f in week_files:
    print(" -", f.name)
print("Usable quote-time cross-sections by day:")
print(pd.Series([c["date"] for c in cross_sections]).value_counts().sort_index().to_string())
print(f"Strikes per cross-section: {len(cross_sections[0]['data'])} = {STRIKES_EACH_SIDE} below + {STRIKES_EACH_SIDE} above")


def heston_cf_log_s(u, S0, tau, v0, kappa, theta, sigma, rho, r=0.0):
    u = np.asarray(u, dtype=np.complex128)
    tau = max(float(tau), 1e-10)
    sigma = max(float(sigma), 1e-8)
    v0 = max(float(v0), 1e-10)
    kappa = max(float(kappa), 1e-8)
    theta = max(float(theta), 1e-10)
    rho = float(np.clip(rho, -0.999, 0.999))

    iu = 1j * u
    d = np.sqrt((rho * sigma * iu - kappa) ** 2 + sigma ** 2 * (iu + u ** 2))
    g = (kappa - rho * sigma * iu - d) / (kappa - rho * sigma * iu + d)
    exp_dt = np.exp(-d * tau)
    log_term = np.log((1.0 - g * exp_dt) / (1.0 - g))

    C = iu * (np.log(S0) + r * tau) + (kappa * theta / sigma ** 2) * (
        (kappa - rho * sigma * iu - d) * tau - 2.0 * log_term
    )
    D = ((kappa - rho * sigma * iu - d) / sigma ** 2) * ((1.0 - exp_dt) / (1.0 - g * exp_dt))
    return np.exp(C + D * v0)


def heston_call_carr_madan(S0, K, tau, params, r=0.0, alpha=CARR_MADAN_ALPHA, n=CARR_MADAN_N, umax=CARR_MADAN_UMAX):
    K = np.asarray(K, dtype=float)
    tau = max(float(tau), 1e-10)
    v0, kappa, theta, sigma, rho = map(float, params)
    umax_eff = min(CARR_MADAN_UMAX_CAP, max(float(umax), 40.0 / np.sqrt(tau)))
    n_from_spacing = int(np.ceil(umax_eff / CARR_MADAN_DU_TARGET)) + 1
    n_floor = 512 if tau * YEAR_SECONDS / 60.0 < 120.0 else int(n)
    n_eff = min(CARR_MADAN_N_MAX, max(int(n), n_floor, n_from_spacing))
    u = np.linspace(1e-8, umax_eff, n_eff)
    du = u[1] - u[0]
    weights = np.full(n_eff, du)
    weights[0] *= 0.5
    weights[-1] *= 0.5

    shifted_u = u - 1j * (alpha + 1.0)
    cf = heston_cf_log_s(shifted_u, S0, tau, v0, kappa, theta, sigma, rho, r)
    denom = alpha ** 2 + alpha - u ** 2 + 1j * (2.0 * alpha + 1.0) * u
    psi = np.exp(-r * tau) * cf / denom

    log_k = np.log(K)
    integrand = np.real(np.exp(-1j * np.outer(log_k, u)) * psi[None, :])
    prices = np.exp(-alpha * log_k) / np.pi * (integrand @ weights)
    return np.maximum(prices, 0.0)


def heston_call_cos(S0, K, tau, params, r=0.0, n=COS_N, trunc_l=COS_L):
    K = np.asarray(K, dtype=float)
    v0, kappa, theta, sigma, rho = map(float, params)
    prices = np.empty_like(K, dtype=float)

    for j, strike in enumerate(K):
        x_center = np.log(S0 / strike) + (r - 0.5 * max(theta, v0)) * tau
        x_width = trunc_l * np.sqrt(max(theta, v0, 1e-8) * max(tau, 1e-10)) + 0.035
        a = min(x_center - x_width, -0.04)
        b = max(x_center + x_width, 0.04)
        c = max(0.0, a)
        d = b
        k_idx = np.arange(n, dtype=float)
        u = k_idx * np.pi / (b - a)

        psi = np.empty(n, dtype=float)
        chi = np.empty(n, dtype=float)
        psi[0] = d - c
        chi[0] = np.exp(d) - np.exp(c)
        uk = u[1:]
        psi[1:] = (np.sin(uk * (d - a)) - np.sin(uk * (c - a))) / uk
        chi[1:] = (
            np.exp(d) * (np.cos(uk * (d - a)) + uk * np.sin(uk * (d - a))) -
            np.exp(c) * (np.cos(uk * (c - a)) + uk * np.sin(uk * (c - a)))
        ) / (1.0 + uk ** 2)

        payoff_coeff = 2.0 / (b - a) * strike * (chi - psi)
        cf_x = heston_cf_log_s(u, S0, tau, v0, kappa, theta, sigma, rho, r) * np.exp(-1j * u * np.log(strike))
        terms = cf_x * np.exp(-1j * u * a) * payoff_coeff
        terms[0] *= 0.5
        prices[j] = np.exp(-r * tau) * np.real(np.sum(terms))

    return np.maximum(prices, 0.0)


def heston_call_broadie_kaya_diagnostic(S0, K, tau, params, r=0.0, n_paths=BK_EVALUATION_PATHS, seed=7):
    K = np.asarray(K, dtype=float)
    v0, kappa, theta, sigma, rho = map(float, params)
    tau = max(float(tau), 1e-10)
    sigma = max(float(sigma), 1e-8)
    kappa = max(float(kappa), 1e-8)
    theta = max(float(theta), 1e-10)
    rho = float(np.clip(rho, -0.999, 0.999))

    local_rng = np.random.default_rng(seed)
    exp_kt = np.exp(-kappa * tau)
    c = sigma ** 2 * (1.0 - exp_kt) / (4.0 * kappa)
    df = 4.0 * kappa * theta / sigma ** 2
    nc = 4.0 * kappa * exp_kt * max(v0, 1e-12) / (sigma ** 2 * (1.0 - exp_kt))
    v_t = c * ncx2.rvs(df, nc, size=n_paths, random_state=local_rng)
    int_v = np.maximum(0.5 * (v0 + v_t) * tau, 1e-12)

    z = local_rng.standard_normal(n_paths)
    variance_shock = (v_t - v0 - kappa * theta * tau + kappa * int_v) / sigma
    log_s = (
        np.log(S0) + r * tau - 0.5 * int_v +
        rho * variance_shock + np.sqrt(max(1.0 - rho ** 2, 1e-10)) * np.sqrt(int_v) * z
    )
    s_t = np.exp(log_s)
    payoffs = np.maximum(s_t[:, None] - K[None, :], 0.0)
    return np.exp(-r * tau) * payoffs.mean(axis=0)


heston_mc_trapezoid_diagnostic = heston_call_broadie_kaya_diagnostic


def bs_call_price(S0, K, tau, sigma, r=0.0):
    S0 = float(S0)
    K = np.asarray(K, dtype=float)
    tau = float(tau)
    sigma = float(sigma)
    if S0 <= 0 or tau <= 0 or sigma <= 0:
        return np.full_like(K, np.nan, dtype=float)
    vol_sqrt_t = sigma * np.sqrt(tau)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * tau) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    return S0 * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)


def implied_vol_call(price, S0, K, tau, r=0.0, vol_low=1e-6, vol_high=8.0, clip_to_bounds=True):
    price = float(price)
    S0 = float(S0)
    K = float(K)
    tau = float(tau)
    if not np.isfinite(price) or S0 <= 0 or K <= 0 or tau <= 0:
        return np.nan
    lower = max(S0 - K * np.exp(-r * tau), 0.0)
    upper = S0
    if clip_to_bounds:
        price = float(np.clip(price, lower + 1e-8, upper - 1e-8))
    elif price < lower - 1e-6 or price > upper + 1e-6:
        return np.nan

    def f(sig):
        return float(bs_call_price(S0, np.array([K]), tau, sig, r)[0] - price)

    try:
        f_low = f(vol_low)
        f_high = f(vol_high)
        while np.isfinite(f_high) and f_low * f_high > 0 and vol_high < 40.0:
            vol_high *= 1.5
            f_high = f(vol_high)
        if not (np.isfinite(f_low) and np.isfinite(f_high)) or f_low * f_high > 0:
            return np.nan
        return float(brentq(f, vol_low, vol_high, xtol=1e-8, maxiter=100))
    except Exception:
        return np.nan


def implied_vol_vector(prices, S0, K, tau, r=0.0, clip_to_bounds=True):
    return np.array([implied_vol_call(px, S0, k, tau, r, clip_to_bounds=clip_to_bounds) for px, k in zip(prices, K)], dtype=float)



def intrinsic_call_value(S0, K, tau, r=0.0):
    K = np.asarray(K, dtype=float)
    return np.maximum(float(S0) - K * np.exp(-float(r) * max(float(tau), 0.0)), 0.0)


def calibration_price_component(prices, S0, K, tau, r=0.0):
    prices = np.asarray(prices, dtype=float)
    intrinsic = intrinsic_call_value(S0, K, tau, r)
    if CALIBRATION_PRICE_MODE == "time_value":
        return np.maximum(prices - intrinsic, 0.0)
    if CALIBRATION_PRICE_MODE == "price":
        return prices
    raise ValueError(f"Unknown CALIBRATION_PRICE_MODE: {CALIBRATION_PRICE_MODE}")


def bounded_array(x):
    lo = np.array([b[0] for b in PARAM_BOUNDS], dtype=float)
    hi = np.array([b[1] for b in PARAM_BOUNDS], dtype=float)
    return np.minimum(np.maximum(np.asarray(x, dtype=float), lo), hi)


def starting_points(previous=None):
    starts = [DEFAULT_START]
    if previous is not None and np.all(np.isfinite(previous)):
        starts.insert(0, bounded_array(previous))
    starts.extend([
        [0.01, 1.0, 0.01, 0.35, -0.25],
        [0.04, 8.0, 0.04, 1.25, -0.70],
        [0.12, 3.0, 0.08, 2.00, -0.40],
        [0.25, 15.0, 0.20, 3.00, 0.10],
    ])
    unique = []
    for s in starts:
        s = bounded_array(s)
        if not any(np.allclose(s, u) for u in unique):
            unique.append(s)
    return unique


def quote_error_scale(market, bid=None, ask=None, reference=None):
    market = np.asarray(market, dtype=float)
    ref = market if reference is None else np.asarray(reference, dtype=float)
    if bid is None or ask is None:
        half_spread = np.zeros_like(market)
    else:
        bid = np.asarray(bid, dtype=float)
        ask = np.asarray(ask, dtype=float)
        half_spread = np.maximum((ask - bid) / 2.0, 0.0)
    return np.maximum.reduce([
        half_spread,
        np.full_like(market, QUOTE_ERROR_FLOOR, dtype=float),
        RELATIVE_ERROR_FLOOR * np.maximum(ref, QUOTE_ERROR_FLOOR),
    ])


def huber_loss(z, delta=HUBER_DELTA):
    z = np.asarray(z, dtype=float)
    abs_z = np.abs(z)
    return np.where(abs_z <= delta, 0.5 * z ** 2, delta * (abs_z - 0.5 * delta))



def bid_ask_band_residual(model_target, bid_target, ask_target, scale):
    above = np.maximum(model_target - ask_target, 0.0)
    below = np.maximum(bid_target - model_target, 0.0)
    return (above - below) / scale


def parameter_distance(x, reference):
    if reference is None or not np.all(np.isfinite(reference)):
        return 0.0
    x = bounded_array(x)
    reference = bounded_array(reference)
    positive = np.log(x[:4]) - np.log(reference[:4])
    rho_x = np.arctanh(np.clip(x[4], -0.995, 0.995))
    rho_ref = np.arctanh(np.clip(reference[4], -0.995, 0.995))
    diff = np.r_[positive, rho_x - rho_ref]
    return float(np.mean(diff ** 2))



def feller_penalty(x):
    v0, kappa, theta, sigma, rho = bounded_array(x)
    violation = max(sigma ** 2 - 2.0 * kappa * theta, 0.0)
    return FELLER_PENALTY_WEIGHT * violation ** 2


def calibration_mode_for_tau(tau_minutes):
    if USE_REDUCED_0DTE_HESTON and tau_minutes < REDUCED_HESTON_TAU_MINUTES:
        return "reduced_v0_sigma_rho"
    return "full"


def fixed_kappa_theta(previous=None):
    if previous is not None and np.all(np.isfinite(previous)):
        return float(previous[1]), float(previous[2])
    return float(FIXED_KAPPA_THETA[0]), float(FIXED_KAPPA_THETA[1])


def expand_reduced_params(z, previous=None):
    kappa, theta = fixed_kappa_theta(previous)
    x = np.array([z[0], kappa, theta, z[1], z[2]], dtype=float)
    return bounded_array(x)


def reduced_starting_points(previous=None):
    starts = []
    if previous is not None and np.all(np.isfinite(previous)):
        starts.append([previous[0], previous[3], previous[4]])
    for full in starting_points(previous):
        starts.append([full[0], full[3], full[4]])
    bounds = [(PARAM_BOUNDS[0][0], PARAM_BOUNDS[0][1]), (PARAM_BOUNDS[3][0], PARAM_BOUNDS[3][1]), (PARAM_BOUNDS[4][0], PARAM_BOUNDS[4][1])]
    lo = np.array([b[0] for b in bounds], dtype=float)
    hi = np.array([b[1] for b in bounds], dtype=float)
    unique = []
    for s in starts:
        z = np.minimum(np.maximum(np.asarray(s, dtype=float), lo), hi)
        if not any(np.allclose(z, u) for u in unique):
            unique.append(z)
    return unique, bounds


def total_variance_errors(model_iv, market_iv, tau):
    model_iv = np.asarray(model_iv, dtype=float)
    market_iv = np.asarray(market_iv, dtype=float)
    tau_arr = np.asarray(tau, dtype=float)
    if tau_arr.ndim == 0:
        tau_arr = np.full_like(model_iv, float(tau_arr), dtype=float)
    valid = (
        np.isfinite(model_iv) & np.isfinite(market_iv) & np.isfinite(tau_arr) & (tau_arr > 0) &
        (model_iv >= TOTAL_VARIANCE_IV_FLOOR) & (model_iv <= TOTAL_VARIANCE_IV_CEIL) &
        (market_iv >= TOTAL_VARIANCE_IV_FLOOR) & (market_iv <= TOTAL_VARIANCE_IV_CEIL)
    )
    err = (model_iv ** 2 - market_iv ** 2) * tau_arr
    return err, valid


def fit_heston_cross_section(S0, K, tau, market, pricing_func, previous=None, calibration_kwargs=None, bid=None, ask=None, tau_minutes=None):
    calibration_kwargs = calibration_kwargs or {}
    tau_minutes = float(tau_minutes if tau_minutes is not None else tau * YEAR_SECONDS / 60.0)
    fit_mode = calibration_mode_for_tau(tau_minutes)
    market_component = calibration_price_component(market, S0, K, tau, RISK_FREE_RATE)
    bid_component = calibration_price_component(bid, S0, K, tau, RISK_FREE_RATE) if bid is not None else market_component
    ask_component = calibration_price_component(ask, S0, K, tau, RISK_FREE_RATE) if ask is not None else market_component
    scale = quote_error_scale(market, bid, ask, reference=market_component)
    history = []

    def loss_for_full_params(x):
        x = bounded_array(x)
        model = pricing_func(S0, K, tau, x, RISK_FREE_RATE, **calibration_kwargs)
        if not np.all(np.isfinite(model)):
            return 1e12
        model_component = calibration_price_component(model, S0, K, tau, RISK_FREE_RATE)
        if CALIBRATION_LOSS_MODE == "bid_ask_band":
            z = bid_ask_band_residual(model_component, bid_component, ask_component, scale)
        else:
            z = (model_component - market_component) / scale
        fit_loss = float(np.mean(huber_loss(z)))
        smooth_penalty = TEMPORAL_REGULARIZATION * parameter_distance(x, previous)
        return fit_loss + smooth_penalty + feller_penalty(x)

    if fit_mode == "reduced_v0_sigma_rho":
        starts, reduced_bounds = reduced_starting_points(previous)

        def objective_reduced(z):
            return loss_for_full_params(expand_reduced_params(z, previous))

        best = None
        for start_id, z0 in enumerate(starts):
            res = minimize(objective_reduced, z0, method="SLSQP", bounds=reduced_bounds, options={"maxiter": OPT_MAXITER_PER_START, "ftol": 1e-7, "disp": False})
            obj = float(res.fun) if np.isfinite(res.fun) else np.inf
            full_x = expand_reduced_params(res.x, previous)
            history.append({"start_id": start_id, "fit_mode": fit_mode, "success": bool(res.success), "objective": obj, **dict(zip(PARAM_NAMES, full_x))})
            if best is None or obj < best[0]:
                best = (obj, res)

        polish = minimize(objective_reduced, best[1].x, method="SLSQP", bounds=reduced_bounds, options={"maxiter": OPT_POLISH_MAXITER, "ftol": 1e-9, "disp": False})
        best_reduced = polish if np.isfinite(polish.fun) and float(polish.fun) <= best[0] * 1.02 else best[1]
        full_x = expand_reduced_params(best_reduced.x, previous)
        best_res = OptimizeResult(x=full_x, fun=float(loss_for_full_params(full_x)), success=bool(best_reduced.success), message=best_reduced.message)
        return best_res, pd.DataFrame(history), fit_mode

    best = None
    for start_id, x0 in enumerate(starting_points(previous)):
        res = minimize(loss_for_full_params, x0, method="SLSQP", bounds=PARAM_BOUNDS, options={"maxiter": OPT_MAXITER_PER_START, "ftol": 1e-7, "disp": False})
        obj = float(res.fun) if np.isfinite(res.fun) else np.inf
        history.append({"start_id": start_id, "fit_mode": fit_mode, "success": bool(res.success), "objective": obj, **dict(zip(PARAM_NAMES, res.x))})
        if best is None or obj < best[0]:
            best = (obj, res)

    polish = minimize(loss_for_full_params, best[1].x, method="SLSQP", bounds=PARAM_BOUNDS, options={"maxiter": OPT_POLISH_MAXITER, "ftol": 1e-9, "disp": False})
    best_res = polish if np.isfinite(polish.fun) and float(polish.fun) <= best[0] * 1.02 else best[1]
    return best_res, pd.DataFrame(history), fit_mode

def run_method_calibration(method_name, pricing_func, calibration_kwargs_factory=None, evaluation_kwargs_factory=None):
    calibration_kwargs_factory = calibration_kwargs_factory or (lambda i, section: {})
    evaluation_kwargs_factory = evaluation_kwargs_factory or calibration_kwargs_factory
    last_params = None
    fit_rows = []
    calib_rows = []
    start_histories = []

    for i, section in enumerate(cross_sections, start=1):
        g = section["data"].sort_values("strike")
        S0 = section["spot"]
        tau = section["tau_years"]
        K = g["strike"].to_numpy(dtype=float)
        market = g["mid"].to_numpy(dtype=float)
        bid = g["bid"].to_numpy(dtype=float)
        ask = g["ask"].to_numpy(dtype=float)
        market_component = calibration_price_component(market, S0, K, tau, RISK_FREE_RATE)
        scale = quote_error_scale(market, bid, ask, reference=market_component)

        res, hist, fit_mode = fit_heston_cross_section(
            S0, K, tau, market, pricing_func, previous=last_params,
            calibration_kwargs=calibration_kwargs_factory(i, section), bid=bid, ask=ask,
            tau_minutes=section["tau_minutes"],
        )
        params = bounded_array(res.x)
        if np.all(np.isfinite(params)):
            last_params = params

        model = pricing_func(S0, K, tau, params, RISK_FREE_RATE, **evaluation_kwargs_factory(i, section))
        market_iv = implied_vol_vector(market, S0, K, tau, RISK_FREE_RATE, clip_to_bounds=True)
        model_iv = implied_vol_vector(model, S0, K, tau, RISK_FREE_RATE, clip_to_bounds=True)
        err = model - market
        model_component = calibration_price_component(model, S0, K, tau, RISK_FREE_RATE)
        component_err = model_component - market_component
        spread_norm_err = component_err / scale
        within = (model >= bid) & (model <= ask)

        hist.insert(0, "quote_datetime", section["quote_datetime"])
        hist.insert(1, "method", method_name)
        start_histories.append(hist)
        row = {
            "quote_datetime": section["quote_datetime"], "date": section["date"], "time": section["time"],
            "minute_of_day": section["minute_of_day"], "spot": S0, "tau_years": tau,
            "tau_minutes": section["tau_minutes"], "n_strikes": len(K), "fit_mode": fit_mode, "success": bool(res.success),
            "objective": float(res.fun) if np.isfinite(res.fun) else np.nan,
            "rmse": float(np.sqrt(np.mean(err ** 2))),
            "mae": float(np.mean(np.abs(err))),
            "time_value_rmse": float(np.sqrt(np.mean(component_err ** 2))),
            "time_value_mae": float(np.mean(np.abs(component_err))),
            "spread_normalized_rmse": float(np.sqrt(np.mean(spread_norm_err ** 2))),
            "mean_abs_spread_normalized_error": float(np.mean(np.abs(spread_norm_err))),
            "within_bid_ask_rate": float(np.mean(within)),
        }
        total_var_err, total_var_valid = total_variance_errors(model_iv, market_iv, tau)
        row["total_variance_rmse"] = float(np.sqrt(np.mean(total_var_err[total_var_valid] ** 2))) if np.any(total_var_valid) else np.nan
        row["total_variance_valid_count"] = int(np.sum(total_var_valid))
        row.update(dict(zip(PARAM_NAMES, params)))
        calib_rows.append(row)

        intrinsic = intrinsic_call_value(S0, K, tau, RISK_FREE_RATE)
        total_var_err, total_var_valid = total_variance_errors(model_iv, market_iv, tau)
        for strike, bid_i, ask_i, mkt, px, mn, miv, piv, scl, zn, inside, intr, mtv, ptv, twe, twv in zip(
            K, bid, ask, market, model, g["moneyness"].to_numpy(dtype=float), market_iv, model_iv,
            scale, spread_norm_err, within, intrinsic, market_component, model_component, total_var_err, total_var_valid
        ):
            fit_rows.append({
                "quote_datetime": section["quote_datetime"], "date": section["date"], "time": section["time"],
                "minute_of_day": section["minute_of_day"], "spot": S0, "tau_years": tau,
                "tau_minutes": section["tau_minutes"], "strike": float(strike), "moneyness": float(mn),
                "bid": float(bid_i), "ask": float(ask_i), "mid": float(mkt),
                "half_spread": float(max((ask_i - bid_i) / 2.0, 0.0)),
                "intrinsic": float(intr),
                "market_time_value": float(mtv),
                f"{method_name}_time_value": float(ptv),
                f"{method_name}_time_value_err": float(ptv - mtv),
                "error_scale": float(scl),
                f"{method_name}_price": float(px),
                "market_iv": float(miv) if np.isfinite(miv) else np.nan,
                f"{method_name}_iv": float(piv) if np.isfinite(piv) else np.nan,
                f"{method_name}_err": float(px - mkt),
                f"{method_name}_abs_err": float(abs(px - mkt)),
                f"{method_name}_rel_err": float((px - mkt) / max(QUOTE_ERROR_FLOOR, mkt)),
                f"{method_name}_spread_norm_err": float(zn),
                f"{method_name}_total_variance_err": float(twe) if np.isfinite(twe) else np.nan,
                f"{method_name}_total_variance_valid": bool(twv),
                f"{method_name}_within_bid_ask": bool(inside),
            })

        if i == 1 or i % 10 == 0 or i == len(cross_sections):
            print(
                f"{method_name}: fit {i:>3}/{len(cross_sections)} | {section['date']} {section['time']} | "
                f"raw RMSE={calib_rows[-1]['rmse']:.4f} | spread-normalized RMSE={calib_rows[-1]['spread_normalized_rmse']:.3f}"
            )

    fit_df = pd.DataFrame(fit_rows)
    calib_df = pd.DataFrame(calib_rows)
    summary = pd.DataFrame([{
        "method": method_name,
        "quote_times": calib_df["quote_datetime"].nunique(),
        "mean_RMSE": calib_df["rmse"].mean(),
        "median_RMSE": calib_df["rmse"].median(),
        "mean_spread_normalized_RMSE": calib_df["spread_normalized_rmse"].mean(),
        "mean_within_bid_ask_rate": calib_df["within_bid_ask_rate"].mean(),
        "mean_total_variance_RMSE": calib_df["total_variance_rmse"].mean(),
        "success_rate": calib_df["success"].mean(),
    }])
    return {"fit": fit_df, "calibration": calib_df, "start_history": pd.concat(start_histories, ignore_index=True), "summary": summary}



def validate_pricers_on_sections(method_params=None, max_sections=PRICER_VALIDATION_MAX_SECTIONS):
    if method_params is None:
        if "fit_results" not in globals() or "carr_madan" not in fit_results:
            raise RuntimeError("Run Carr-Madan calibration first or pass method_params.")
        method_params = fit_results["carr_madan"]["calibration"]

    rows = []
    sample_sections = cross_sections[:max_sections]
    for section in sample_sections:
        qdt = section["quote_datetime"]
        params_row = method_params.loc[method_params["quote_datetime"] == qdt]
        if params_row.empty:
            continue
        params = params_row[PARAM_NAMES].iloc[0].to_numpy(dtype=float)
        g = section["data"].sort_values("strike")
        S0 = section["spot"]
        tau = section["tau_years"]
        K = g["strike"].to_numpy(dtype=float)
        cm = heston_call_carr_madan(S0, K, tau, params, RISK_FREE_RATE)
        co = heston_call_cos(S0, K, tau, params, RISK_FREE_RATE)
        rows.append({
            "quote_datetime": qdt,
            "date": section["date"],
            "time": section["time"],
            "tau_minutes": section["tau_minutes"],
            "max_abs_carr_madan_minus_cos": float(np.max(np.abs(cm - co))),
            "rmse_carr_madan_minus_cos": float(np.sqrt(np.mean((cm - co) ** 2))),
            "n_strikes": len(K),
        })
    return pd.DataFrame(rows)


fit_results = {}


Data directory: C:\Users\wojte\Desktop\magisterka\2022_0dte_spy\2022
Using one full true 0DTE week:
 - spx-options-2022-05-16.gzip.parquet
 - spx-options-2022-05-17.gzip.parquet
 - spx-options-2022-05-18.gzip.parquet
 - spx-options-2022-05-19.gzip.parquet
 - spx-options-2022-05-20.gzip.parquet
Usable quote-time cross-sections by day:
2022-05-16     9
2022-05-17    10
2022-05-18    11
2022-05-19    11
2022-05-20    11
Strikes per cross-section: 20 = 10 below + 10 above


In [2]:
# Cell 2: compiler/pricing method 1 - Carr-Madan.
fit_results["carr_madan"] = run_method_calibration(
    "carr_madan",
    heston_call_carr_madan,
    calibration_kwargs_factory=lambda i, section: {"n": CARR_MADAN_N, "umax": CARR_MADAN_UMAX, "alpha": CARR_MADAN_ALPHA},
    evaluation_kwargs_factory=lambda i, section: {"n": CARR_MADAN_N, "umax": CARR_MADAN_UMAX, "alpha": CARR_MADAN_ALPHA},
)
display(fit_results["carr_madan"]["summary"])
display(fit_results["carr_madan"]["calibration"].head())


carr_madan: fit   1/52 | 2022-05-16 09:31 | raw RMSE=0.2346 | spread-normalized RMSE=0.386
carr_madan: fit  10/52 | 2022-05-17 09:31 | raw RMSE=0.1875 | spread-normalized RMSE=0.546
carr_madan: fit  20/52 | 2022-05-18 09:31 | raw RMSE=0.1724 | spread-normalized RMSE=0.369
carr_madan: fit  30/52 | 2022-05-18 15:30 | raw RMSE=0.1484 | spread-normalized RMSE=1.339
carr_madan: fit  40/52 | 2022-05-19 15:00 | raw RMSE=0.4402 | spread-normalized RMSE=0.894
carr_madan: fit  50/52 | 2022-05-20 14:30 | raw RMSE=0.1104 | spread-normalized RMSE=0.564
carr_madan: fit  52/52 | 2022-05-20 15:30 | raw RMSE=0.1505 | spread-normalized RMSE=0.826


,method,quote_times,mean_RMSE,median_RMSE,mean_spread_normalized_RMSE,mean_within_bid_ask_rate,mean_total_variance_RMSE,success_rate
0,carr_madan,52,0.272509,0.237733,0.695833,0.7875,0.000006,1.0


,quote_datetime,date,time,minute_of_day,spot,tau_years,tau_minutes,n_strikes,fit_mode,success,...,spread_normalized_rmse,mean_abs_spread_normalized_error,within_bid_ask_rate,total_variance_rmse,total_variance_valid_count,v0,kappa,theta,sigma,rho
0,2022-05-16 09:31:00-04:00,2022-05-16,09:31,571,4010.9199,0.000740,389.0,20,full,True,...,0.385902,0.310358,1.0,0.000005,20,0.307916,15.001620,0.295516,2.977659,-0.095505
1,2022-05-16 10:00:00-04:00,2022-05-16,10:00,600,3993.9800,0.000684,360.0,20,full,True,...,0.345728,0.275755,1.0,0.000004,20,0.334250,15.004132,0.296080,2.979969,0.041991
2,2022-05-16 10:30:00-04:00,2022-05-16,10:30,630,4023.7700,0.000627,330.0,20,full,True,...,0.650601,0.580922,0.9,0.000007,20,0.281347,15.026266,0.491973,3.854242,-0.703748
3,2022-05-16 11:00:00-04:00,2022-05-16,11:00,660,3992.7900,0.000570,300.0,20,full,True,...,0.395795,0.312425,0.9,0.000002,20,0.253367,15.003403,0.492908,2.986366,-0.239580
4,2022-05-16 12:30:00-04:00,2022-05-16,12:30,750,4011.2400,0.000399,210.0,20,full,True,...,0.634655,0.567204,0.9,0.000004,20,0.220768,15.000431,0.492705,3.841084,-0.538925


In [3]:
# Cell 3: compiler/pricing method 2 - COS expansion.
fit_results["cos"] = run_method_calibration(
    "cos",
    heston_call_cos,
    calibration_kwargs_factory=lambda i, section: {"n": COS_N, "trunc_l": COS_L},
    evaluation_kwargs_factory=lambda i, section: {"n": COS_N, "trunc_l": COS_L},
)
display(fit_results["cos"]["summary"])
display(fit_results["cos"]["calibration"].head())


cos: fit   1/52 | 2022-05-16 09:31 | raw RMSE=0.2346 | spread-normalized RMSE=0.386
cos: fit  10/52 | 2022-05-17 09:31 | raw RMSE=0.1189 | spread-normalized RMSE=0.316
cos: fit  20/52 | 2022-05-18 09:31 | raw RMSE=0.1466 | spread-normalized RMSE=0.353
cos: fit  30/52 | 2022-05-18 15:30 | raw RMSE=0.1586 | spread-normalized RMSE=1.380
cos: fit  40/52 | 2022-05-19 15:00 | raw RMSE=0.4403 | spread-normalized RMSE=0.894
cos: fit  50/52 | 2022-05-20 14:30 | raw RMSE=0.1087 | spread-normalized RMSE=0.570
cos: fit  52/52 | 2022-05-20 15:30 | raw RMSE=0.1507 | spread-normalized RMSE=0.827


,method,quote_times,mean_RMSE,median_RMSE,mean_spread_normalized_RMSE,mean_within_bid_ask_rate,mean_total_variance_RMSE,success_rate
0,cos,52,0.26965,0.241932,0.696376,0.789423,0.000006,1.0


,quote_datetime,date,time,minute_of_day,spot,tau_years,tau_minutes,n_strikes,fit_mode,success,...,spread_normalized_rmse,mean_abs_spread_normalized_error,within_bid_ask_rate,total_variance_rmse,total_variance_valid_count,v0,kappa,theta,sigma,rho
0,2022-05-16 09:31:00-04:00,2022-05-16,09:31,571,4010.9199,0.000740,389.0,20,full,True,...,0.385948,0.310403,1.0,0.000005,20,0.307920,15.001830,0.295548,2.977839,-0.095397
1,2022-05-16 10:00:00-04:00,2022-05-16,10:00,600,3993.9800,0.000684,360.0,20,full,True,...,0.346403,0.276487,1.0,0.000004,20,0.334265,15.005449,0.296171,2.980662,0.043132
2,2022-05-16 10:30:00-04:00,2022-05-16,10:30,630,4023.7700,0.000627,330.0,20,full,True,...,0.650704,0.581016,0.9,0.000007,20,0.281350,15.024753,0.492241,3.854645,-0.703548
3,2022-05-16 11:00:00-04:00,2022-05-16,11:00,660,3992.7900,0.000570,300.0,20,full,True,...,0.395546,0.312263,0.9,0.000002,20,0.253379,15.003331,0.489809,2.987395,-0.238949
4,2022-05-16 12:30:00-04:00,2022-05-16,12:30,750,4011.2400,0.000399,210.0,20,full,True,...,0.634647,0.567097,0.9,0.000004,20,0.220783,15.000221,0.490998,3.837255,-0.539360


In [4]:
# Cell 4: compiler/pricing method 3 - Broadie-Kaya-style Monte Carlo diagnostic, fitted on its own.
# This calibrates its own Heston parameters with fixed seeds for deterministic optimizer evaluations.
# Interpret it as a noisy diagnostic compiler, not as the primary production calibration engine.
fit_results["broadie_kaya"] = run_method_calibration(
    "broadie_kaya",
    heston_mc_trapezoid_diagnostic,
    calibration_kwargs_factory=lambda i, section: {"n_paths": BK_CALIBRATION_PATHS, "seed": 17000 + i},
    evaluation_kwargs_factory=lambda i, section: {"n_paths": BK_EVALUATION_PATHS, "seed": 27000 + i},
)

display(fit_results["broadie_kaya"]["summary"])
display(fit_results["broadie_kaya"]["calibration"].head())


broadie_kaya: fit   1/52 | 2022-05-16 09:31 | raw RMSE=1.7948 | spread-normalized RMSE=4.314
broadie_kaya: fit  10/52 | 2022-05-17 09:31 | raw RMSE=0.6251 | spread-normalized RMSE=1.495
broadie_kaya: fit  20/52 | 2022-05-18 09:31 | raw RMSE=0.1372 | spread-normalized RMSE=0.538
broadie_kaya: fit  30/52 | 2022-05-18 15:30 | raw RMSE=0.5616 | spread-normalized RMSE=2.703
broadie_kaya: fit  40/52 | 2022-05-19 15:00 | raw RMSE=0.1943 | spread-normalized RMSE=1.458
broadie_kaya: fit  50/52 | 2022-05-20 14:30 | raw RMSE=0.9222 | spread-normalized RMSE=3.651
broadie_kaya: fit  52/52 | 2022-05-20 15:30 | raw RMSE=0.2480 | spread-normalized RMSE=1.327


,method,quote_times,mean_RMSE,median_RMSE,mean_spread_normalized_RMSE,mean_within_bid_ask_rate,mean_total_variance_RMSE,success_rate
0,broadie_kaya,52,0.730186,0.638763,2.846439,0.283654,0.000014,0.980769


,quote_datetime,date,time,minute_of_day,spot,tau_years,tau_minutes,n_strikes,fit_mode,success,...,spread_normalized_rmse,mean_abs_spread_normalized_error,within_bid_ask_rate,total_variance_rmse,total_variance_valid_count,v0,kappa,theta,sigma,rho
0,2022-05-16 09:31:00-04:00,2022-05-16,09:31,571,4010.9199,0.000740,389.0,20,full,True,...,4.313599,4.097666,0.00,0.000036,20,0.263482,3.317676,0.995852,2.300353,0.823691
1,2022-05-16 10:00:00-04:00,2022-05-16,10:00,600,3993.9800,0.000684,360.0,20,full,True,...,1.544571,1.464244,0.15,0.000016,20,0.326737,15.000445,0.266174,2.993021,0.075391
2,2022-05-16 10:30:00-04:00,2022-05-16,10:30,630,4023.7700,0.000627,330.0,20,full,True,...,5.739340,5.400402,0.00,0.000039,20,0.224696,3.009233,0.311341,1.935875,-0.348633
3,2022-05-16 11:00:00-04:00,2022-05-16,11:00,660,3992.7900,0.000570,300.0,20,full,True,...,1.009810,0.914135,0.35,0.000009,20,0.238918,3.045209,0.330228,1.988248,-0.339701
4,2022-05-16 12:30:00-04:00,2022-05-16,12:30,750,4011.2400,0.000399,210.0,20,full,True,...,4.524310,4.194008,0.05,0.000015,20,0.185643,3.030549,0.341125,2.065961,-0.491472


In [5]:
# Cell 5: plotting through time, price/IV surfaces, and errors to mid.
METHODS = [m for m in ["carr_madan", "cos", "broadie_kaya"] if m in fit_results]
missing = [m for m in METHODS if m not in fit_results]
if missing:
    raise RuntimeError(f"Run the method cells first. Missing: {missing}")

base_cols = [
    "quote_datetime", "date", "time", "minute_of_day", "spot", "tau_years", "tau_minutes",
    "strike", "moneyness", "bid", "ask", "mid", "half_spread", "intrinsic", "market_time_value", "error_scale", "market_iv",
]
method_cols = lambda m: [
    f"{m}_price", f"{m}_iv", f"{m}_err", f"{m}_abs_err", f"{m}_rel_err",
    f"{m}_time_value", f"{m}_time_value_err", f"{m}_spread_norm_err",
    f"{m}_total_variance_err", f"{m}_total_variance_valid", f"{m}_within_bid_ask",
]
combined_fit = fit_results[METHODS[0]]["fit"][base_cols + method_cols(METHODS[0])].copy()
merge_keys = ["quote_datetime", "strike"]
for method in METHODS[1:]:
    cols = merge_keys + method_cols(method)
    combined_fit = combined_fit.merge(fit_results[method]["fit"][cols], on=merge_keys, how="left")

summary = pd.concat([fit_results[m]["summary"] for m in METHODS], ignore_index=True)
display(summary)

validation_df = validate_pricers_on_sections(fit_results["carr_madan"]["calibration"] if "carr_madan" in fit_results else None)
display(validation_df)



def build_fit_diagnostics(combined_fit, methods=METHODS):
    rows = []
    for method in methods:
        tmp = combined_fit[[
            "quote_datetime", "date", "time", "tau_minutes", "strike", "moneyness", "mid",
            "market_time_value", "half_spread", "error_scale",
            f"{method}_err", f"{method}_time_value_err", f"{method}_spread_norm_err",
            f"{method}_within_bid_ask",
        ]].copy()
        tmp["method"] = method
        tmp = tmp.rename(columns={
            f"{method}_err": "raw_err",
            f"{method}_time_value_err": "time_value_err",
            f"{method}_spread_norm_err": "spread_norm_err",
            f"{method}_within_bid_ask": "within_bid_ask",
        })
        rows.append(tmp)
    out = pd.concat(rows, ignore_index=True)
    out["abs_spread_norm_err"] = out["spread_norm_err"].abs()
    out["tau_bucket"] = pd.cut(
        out["tau_minutes"],
        bins=[0, 30, 60, 120, 240, np.inf],
        labels=["0-30m", "30-60m", "60-120m", "120-240m", "240m+"],
        include_lowest=True,
    )
    out["moneyness_bucket"] = pd.cut(
        out["moneyness"],
        bins=[0.0, 0.985, 0.995, 1.005, 1.015, np.inf],
        labels=["deep_ITM_call", "ITM_call", "ATM", "OTM_call", "deep_OTM_call"],
        include_lowest=True,
    )
    return out


fit_diagnostics = build_fit_diagnostics(combined_fit)
error_by_tau_bucket = (
    fit_diagnostics.groupby(["method", "tau_bucket"], observed=True)
    .agg(
        n=("spread_norm_err", "size"),
        spread_norm_rmse=("spread_norm_err", lambda x: float(np.sqrt(np.mean(np.asarray(x) ** 2)))),
        mean_abs_spread_norm_err=("abs_spread_norm_err", "mean"),
        bid_ask_hit_rate=("within_bid_ask", "mean"),
        raw_rmse=("raw_err", lambda x: float(np.sqrt(np.mean(np.asarray(x) ** 2)))),
    )
    .reset_index()
)
error_by_moneyness_bucket = (
    fit_diagnostics.groupby(["method", "moneyness_bucket"], observed=True)
    .agg(
        n=("spread_norm_err", "size"),
        spread_norm_rmse=("spread_norm_err", lambda x: float(np.sqrt(np.mean(np.asarray(x) ** 2)))),
        mean_abs_spread_norm_err=("abs_spread_norm_err", "mean"),
        bid_ask_hit_rate=("within_bid_ask", "mean"),
        raw_rmse=("raw_err", lambda x: float(np.sqrt(np.mean(np.asarray(x) ** 2)))),
    )
    .reset_index()
)
parameter_path = pd.concat(
    [fit_results[m]["calibration"].assign(method=m) for m in METHODS],
    ignore_index=True,
)
bid_ask_violation_rows = fit_diagnostics[~fit_diagnostics["within_bid_ask"]].copy()

display(error_by_tau_bucket)
display(error_by_moneyness_bucket)
display(parameter_path[["method", "quote_datetime", "tau_minutes", "fit_mode", *PARAM_NAMES, "spread_normalized_rmse", "within_bid_ask_rate", "total_variance_rmse"]].head())
display(bid_ask_violation_rows.head())


def plot_rmse_through_time(metric="spread_normalized_rmse"):
    fig = go.Figure()
    for method in METHODS:
        c = fit_results[method]["calibration"].sort_values("quote_datetime")
        fig.add_trace(go.Scatter(x=c["quote_datetime"], y=c[metric], mode="lines+markers", name=method))
    y_title = "Spread-normalized RMSE" if metric == "spread_normalized_rmse" else "Raw RMSE to mid"
    fig.update_layout(title=f"Calibration {y_title} through time", xaxis_title="Quote time", yaxis_title=y_title, template="plotly_white", height=450)
    fig.show()


def plot_bid_ask_hit_rate():
    fig = go.Figure()
    for method in METHODS:
        c = fit_results[method]["calibration"].sort_values("quote_datetime")
        fig.add_trace(go.Scatter(x=c["quote_datetime"], y=c["within_bid_ask_rate"], mode="lines+markers", name=method))
    fig.update_layout(title="Fraction of model prices inside bid/ask through time", xaxis_title="Quote time", yaxis_title="inside bid/ask rate", template="plotly_white", height=430, yaxis=dict(range=[0, 1]))
    fig.show()


def plot_parameter_through_time(param):
    fig = go.Figure()
    for method in METHODS:
        c = fit_results[method]["calibration"].sort_values("quote_datetime")
        fig.add_trace(go.Scatter(x=c["quote_datetime"], y=c[param], mode="lines+markers", name=method))
    fig.update_layout(title=f"Heston parameter {param} through time", xaxis_title="Quote time", yaxis_title=param, template="plotly_white", height=420)
    fig.show()



def option_side_region_traces_for_surface(df, pivot, value_col):
    spot_by_time = df.groupby("quote_datetime")["spot"].median().reindex(pivot.index)
    strikes = pivot.columns.astype(float).to_numpy()
    strike_min = float(np.nanmin(strikes))
    strike_max = float(np.nanmax(strikes))

    z_values = pivot.to_numpy(dtype=float)
    finite_z = z_values[np.isfinite(z_values)]
    if finite_z.size == 0:
        z_min, z_max = -1.0, 1.0
    else:
        z_min, z_max = float(finite_z.min()), float(finite_z.max())
        if np.isclose(z_min, z_max):
            pad = max(abs(z_min) * 0.10, 1e-8)
            z_min -= pad
            z_max += pad
    z_span = max(z_max - z_min, 1e-8)
    z_floor = z_min - 0.04 * z_span

    y_labels = [str(t) for t in pivot.index]
    y = np.column_stack([y_labels, y_labels])
    spot = spot_by_time.to_numpy(dtype=float)
    n = len(pivot.index)
    gradient = np.tile(np.linspace(0.0, 1.0, 2), (n, 1))

    put_x = np.column_stack([np.full(n, strike_min), spot])
    put_z = np.full((n, 2), z_floor, dtype=float)
    call_x = np.column_stack([spot, np.full(n, strike_max)])
    call_z = np.full((n, 2), z_floor, dtype=float)
    atm_x = np.column_stack([spot, spot])
    atm_z = np.column_stack([np.full(n, z_min), np.full(n, z_max)])

    return [
        go.Surface(
            x=put_x, y=y, z=put_z,
            surfacecolor=gradient,
            colorscale=[[0.0, "rgb(0,20,255)"], [1.0, "rgb(0,255,255)"]],
            opacity=0.36,
            showscale=False,
            name="Synthetic put side (K < spot)",
            hovertemplate="synthetic put side<br>K=%{x:.0f}<br>time=%{y}<extra></extra>",
        ),
        go.Surface(
            x=call_x, y=y, z=call_z,
            surfacecolor=gradient,
            colorscale=[[0.0, "rgb(255,20,20)"], [1.0, "rgb(255,235,0)"]],
            opacity=0.36,
            showscale=False,
            name="Call side (K > spot)",
            hovertemplate="call side<br>K=%{x:.0f}<br>time=%{y}<extra></extra>",
        ),
        go.Surface(
            x=atm_x, y=y, z=atm_z,
            surfacecolor=np.zeros_like(atm_z, dtype=float),
            colorscale=[[0.0, "rgb(0,0,0)"], [1.0, "rgb(0,0,0)"]],
            opacity=0.45,
            showscale=False,
            name="Underlying / ATM divider",
            hovertemplate="spot=%{x:.2f}<br>time=%{y}<extra>Underlying divider</extra>",
        ),
    ]


def plot_surface(value_col, title, day=None, colorscale="Viridis"):
    df = combined_fit if day is None else combined_fit[combined_fit["date"] == day]
    pivot = df.pivot_table(index="quote_datetime", columns="strike", values=value_col, aggfunc="mean").sort_index()
    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=pivot.columns.astype(float),
        y=[str(x) for x in pivot.index],
        z=pivot.to_numpy(dtype=float),
        colorscale=colorscale,
        colorbar=dict(title=value_col),
        name=value_col,
    ))
    for trace in option_side_region_traces_for_surface(df, pivot, value_col):
        fig.add_trace(trace)
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="Strike", yaxis_title="Quote time", zaxis_title=value_col),
        template="plotly_white",
        height=720,
    )
    fig.show()

def plot_error_box(error_kind="spread_norm"):
    fig = go.Figure()
    for method in METHODS:
        col = f"{method}_spread_norm_err" if error_kind == "spread_norm" else f"{method}_err"
        fig.add_trace(go.Box(y=combined_fit[col], name=method, boxmean=True))
    title = "Spread-normalized model error to mid" if error_kind == "spread_norm" else "Raw model price error to mid"
    y_title = "(model - mid) / quote error scale" if error_kind == "spread_norm" else "model price - mid"
    fig.update_layout(title=title, yaxis_title=y_title, template="plotly_white", height=450)
    fig.show()


plot_rmse_through_time("spread_normalized_rmse")
plot_rmse_through_time("rmse")
plot_bid_ask_hit_rate()
for p in PARAM_NAMES:
    plot_parameter_through_time(p)

first_day = combined_fit["date"].min()
plot_surface("mid", f"Market mid price surface - {first_day}", day=first_day)
plot_surface("market_time_value", f"Market time-value surface - {first_day}", day=first_day)
for method in METHODS:
    plot_surface(f"{method}_price", f"{method} fitted price surface - {first_day}", day=first_day)
    plot_surface(f"{method}_time_value", f"{method} fitted time-value surface - {first_day}", day=first_day)
    plot_surface(f"{method}_spread_norm_err", f"{method} spread-normalized error surface - {first_day}", day=first_day, colorscale="RdBu")
    plot_surface(f"{method}_err", f"{method} raw error to mid surface - {first_day}", day=first_day, colorscale="RdBu")
plot_error_box("spread_norm")
plot_error_box("raw")


,method,quote_times,mean_RMSE,median_RMSE,mean_spread_normalized_RMSE,mean_within_bid_ask_rate,mean_total_variance_RMSE,success_rate
0,carr_madan,52,0.272509,0.237733,0.695833,0.787500,0.000006,1.000000
1,cos,52,0.269650,0.241932,0.696376,0.789423,0.000006,1.000000
2,broadie_kaya,52,0.730186,0.638763,2.846439,0.283654,0.000014,0.980769


,quote_datetime,date,time,tau_minutes,max_abs_carr_madan_minus_cos,rmse_carr_madan_minus_cos,n_strikes
0,2022-05-16 09:31:00-04:00,2022-05-16,09:31,389.0,0.000168,0.000168,20
1,2022-05-16 10:00:00-04:00,2022-05-16,10:00,360.0,0.000168,0.000167,20
2,2022-05-16 10:30:00-04:00,2022-05-16,10:30,330.0,0.000169,0.000169,20
3,2022-05-16 11:00:00-04:00,2022-05-16,11:00,300.0,0.000167,0.000167,20
4,2022-05-16 12:30:00-04:00,2022-05-16,12:30,210.0,0.000168,0.000168,20


,method,tau_bucket,n,spread_norm_rmse,mean_abs_spread_norm_err,bid_ask_hit_rate,raw_rmse
0,broadie_kaya,0-30m,60,1.945074,1.433206,0.483333,0.449406
1,broadie_kaya,30-60m,80,3.109592,2.291997,0.400000,0.728111
2,broadie_kaya,60-120m,200,4.720202,3.914605,0.195000,0.980270
3,broadie_kaya,120-240m,300,3.160810,2.457642,0.226667,0.683758
4,broadie_kaya,240m+,400,2.689383,1.966685,0.317500,0.962415
5,carr_madan,0-30m,60,0.994728,0.727505,0.683333,0.251722
6,carr_madan,30-60m,80,1.125038,0.860791,0.625000,0.369545
7,carr_madan,60-120m,200,0.982153,0.797619,0.645000,0.379490
8,carr_madan,120-240m,300,0.671241,0.546273,0.786667,0.256546
9,carr_madan,240m+,400,0.531194,0.428360,0.907500,0.291774


,method,moneyness_bucket,n,spread_norm_rmse,mean_abs_spread_norm_err,bid_ask_hit_rate,raw_rmse
0,broadie_kaya,deep_ITM_call,10,0.429015,0.317251,0.900000,0.750536
1,broadie_kaya,ITM_call,306,2.337068,1.542930,0.535948,1.087251
2,broadie_kaya,ATM,409,3.388196,2.666787,0.171149,0.875052
3,broadie_kaya,OTM_call,315,3.975524,3.207030,0.165079,0.494196
4,carr_madan,deep_ITM_call,10,0.430244,0.348992,1.000000,0.846763
5,carr_madan,ITM_call,306,0.677032,0.543885,0.866013,0.471964
6,carr_madan,ATM,409,0.683379,0.533695,0.782396,0.204897
7,carr_madan,OTM_call,315,0.923190,0.695439,0.711111,0.123757
8,cos,deep_ITM_call,10,0.404194,0.308670,1.000000,0.796271
9,cos,ITM_call,306,0.673479,0.541354,0.862745,0.467316


,method,quote_datetime,tau_minutes,fit_mode,v0,kappa,theta,sigma,rho,spread_normalized_rmse,within_bid_ask_rate,total_variance_rmse
0,carr_madan,2022-05-16 09:31:00-04:00,389.0,full,0.307916,15.001620,0.295516,2.977659,-0.095505,0.385902,1.0,0.000005
1,carr_madan,2022-05-16 10:00:00-04:00,360.0,full,0.334250,15.004132,0.296080,2.979969,0.041991,0.345728,1.0,0.000004
2,carr_madan,2022-05-16 10:30:00-04:00,330.0,full,0.281347,15.026266,0.491973,3.854242,-0.703748,0.650601,0.9,0.000007
3,carr_madan,2022-05-16 11:00:00-04:00,300.0,full,0.253367,15.003403,0.492908,2.986366,-0.239580,0.395795,0.9,0.000002
4,carr_madan,2022-05-16 12:30:00-04:00,210.0,full,0.220768,15.000431,0.492705,3.841084,-0.538925,0.634655,0.9,0.000004


,quote_datetime,date,time,tau_minutes,strike,moneyness,mid,market_time_value,half_spread,error_scale,raw_err,time_value_err,spread_norm_err,within_bid_ask,method,abs_spread_norm_err,tau_bucket,moneyness_bucket
42,2022-05-16 10:30:00-04:00,2022-05-16,10:30,330.0,3985.0,0.990365,47.10,8.33,0.70,0.7000,-0.707968,-0.707968,-1.011383,False,carr_madan,1.011383,240m+,ITM_call
55,2022-05-16 10:30:00-04:00,2022-05-16,10:30,330.0,4050.0,1.006519,10.40,10.40,0.10,0.2080,0.100589,0.100589,0.483601,False,carr_madan,0.483601,240m+,OTM_call
69,2022-05-16 11:00:00-04:00,2022-05-16,11:00,300.0,3990.0,0.999301,20.45,17.66,0.15,0.3532,0.151054,0.151054,0.427672,False,carr_madan,0.427672,240m+,ATM
78,2022-05-16 11:00:00-04:00,2022-05-16,11:00,300.0,4035.0,1.010572,5.10,5.10,0.10,0.1020,-0.100126,-0.100126,-0.981632,False,carr_madan,0.981632,240m+,OTM_call
83,2022-05-16 12:30:00-04:00,2022-05-16,12:30,210.0,3980.0,0.992212,36.20,4.96,0.50,0.5000,-0.501072,-0.501072,-1.002145,False,carr_madan,1.002145,120-240m,ITM_call


# Black-Scholes Comparison and Greeks

This section compares the fitted Heston-style methods against a Black-Scholes benchmark and computes Greek surfaces from Black-Scholes implied volatility.

## Black-Scholes Model

For each option row:

- $S$: underlying spot.
- $K$: strike.
- $\tau$: time to expiry in years.
- $r$: risk-free rate.
- $\sigma$: Black-Scholes volatility.
- $\Phi(\cdot)$: standard normal CDF.
- $\phi(\cdot)$: standard normal PDF.

Define

$$d_1=\frac{\log(S/K)+(r+\frac{1}{2}\sigma^2)\tau}{\sigma\sqrt{\tau}}$$

$$d_2=d_1-\sigma\sqrt{\tau}$$

Call price:

$$C^{BS}=S\Phi(d_1)-Ke^{-r\tau}\Phi(d_2)$$

Put price:

$$P^{BS}=Ke^{-r\tau}\Phi(-d_2)-S\Phi(-d_1)$$

The benchmark fits one Black-Scholes volatility per quote-time cross-section using the same time-value / bid-ask-band robust objective as the Heston methods. This makes the comparison fair: all models are judged on the same transformed 0DTE target.

## Implied Volatility

Market implied volatility $\sigma_{\mathrm{IV}}$ solves

$$C^{BS}(S,K,\tau,r,\sigma_{\mathrm{IV}})=M$$

where $M$ is the observed market mid. Numerical root finding is used, and invalid or arbitrage-violating rows return missing values.

## Greeks

The notebook computes Greeks by finite differences of the Black-Scholes price function, using market implied volatility per row when available and falling back to the fitted cross-section Black-Scholes volatility.

First-order Greeks:

$$\Delta=\frac{\partial V}{\partial S}$$

$$\nu=\frac{\partial V}{\partial \sigma}$$

$$\Theta=\frac{\partial V}{\partial t}=-\frac{\partial V}{\partial \tau}$$

$$\rho_r=\frac{\partial V}{\partial r}$$

Second-order and higher-order Greeks:

$$\Gamma=\frac{\partial^2V}{\partial S^2}$$

$$\mathrm{Vanna}=\frac{\partial^2V}{\partial S\,\partial\sigma}$$

$$\mathrm{Vomma}=\frac{\partial^2V}{\partial\sigma^2}$$

$$\mathrm{Speed}=\frac{\partial^3V}{\partial S^3}$$

$$\mathrm{Charm}=\frac{\partial\Delta}{\partial t}$$

$$\mathrm{Colour}=\frac{\partial\Gamma}{\partial t}$$

$$\mathrm{Zomma}=\frac{\partial\Gamma}{\partial\sigma}$$

$$\mathrm{Ultima}=\frac{\partial^3V}{\partial\sigma^3}$$

## Greek 3D Fields

The Greek plots are 3D fields:

- x-axis: strike $K$,
- y-axis: quote time,
- z-axis: Greek value.

The dropdown switches between delta, gamma, vega, theta, rho-rate, vanna, vomma, speed, charm, colour, zomma, and ultima.

The same high-contrast side sheets are shown:

- blue/cyan: synthetic put side, $K<S_t$,
- red/yellow: call side, $K>S_t$,
- black: underlying / ATM divider, $K=S_t$.


In [6]:
# Cell 6: compare independently fitted Heston methods against fitted Black-Scholes.
if "combined_fit" not in globals():
    raise RuntimeError("Run the plotting/combination cell first so combined_fit exists.")

bs_rows = []
for section in cross_sections:
    g = section["data"].sort_values("strike")
    S0 = section["spot"]
    tau = section["tau_years"]
    K = g["strike"].to_numpy(dtype=float)
    market = g["mid"].to_numpy(dtype=float)
    bid = g["bid"].to_numpy(dtype=float)
    ask = g["ask"].to_numpy(dtype=float)
    market_component = calibration_price_component(market, S0, K, tau, RISK_FREE_RATE)
    bid_component = calibration_price_component(bid, S0, K, tau, RISK_FREE_RATE)
    ask_component = calibration_price_component(ask, S0, K, tau, RISK_FREE_RATE)
    scale = quote_error_scale(market, bid, ask, reference=market_component)

    def bs_objective(sig):
        model = bs_call_price(S0, K, tau, sig, RISK_FREE_RATE)
        model_component = calibration_price_component(model, S0, K, tau, RISK_FREE_RATE)
        if CALIBRATION_LOSS_MODE == "bid_ask_band":
            z = bid_ask_band_residual(model_component, bid_component, ask_component, scale)
        else:
            z = (model_component - market_component) / scale
        return float(np.mean(huber_loss(z)))

    res = minimize_scalar(bs_objective, bounds=(1e-5, 8.0), method="bounded", options={"xatol": 1e-6})
    sigma_bs = float(res.x)
    prices = bs_call_price(S0, K, tau, sigma_bs, RISK_FREE_RATE)
    ivs = implied_vol_vector(prices, S0, K, tau, RISK_FREE_RATE, clip_to_bounds=True)

    intrinsic = intrinsic_call_value(S0, K, tau, RISK_FREE_RATE)
    bs_component = calibration_price_component(prices, S0, K, tau, RISK_FREE_RATE)
    for strike, bid_i, ask_i, mid, price, iv, scl, intr, mtv, btv in zip(K, bid, ask, market, prices, ivs, scale, intrinsic, market_component, bs_component):
        bs_rows.append({
            "quote_datetime": section["quote_datetime"],
            "strike": float(strike),
            "bs_sigma_fit": sigma_bs,
            "bs_price": float(price),
            "bs_iv": float(iv) if np.isfinite(iv) else np.nan,
            "bs_time_value": float(btv),
            "bs_time_value_err": float(btv - mtv),
            "bs_err": float(price - mid),
            "bs_abs_err": float(abs(price - mid)),
            "bs_rel_err": float((price - mid) / max(QUOTE_ERROR_FLOOR, mid)),
            "bs_spread_norm_err": float((btv - mtv) / scl),
            "bs_within_bid_ask": bool((price >= bid_i) and (price <= ask_i)),
        })

bs_fit = pd.DataFrame(bs_rows)
comparison_df = combined_fit.merge(bs_fit, on=["quote_datetime", "strike"], how="left")

comparison_summary = []
for method in METHODS + ["bs"]:
    price_col = "bs_price" if method == "bs" else f"{method}_price"
    spread_col = "bs_spread_norm_err" if method == "bs" else f"{method}_spread_norm_err"
    inside_col = "bs_within_bid_ask" if method == "bs" else f"{method}_within_bid_ask"
    err = comparison_df[price_col] - comparison_df["mid"]
    if method == "bs":
        tw_err, tw_valid = total_variance_errors(comparison_df["bs_iv"], comparison_df["market_iv"], comparison_df["tau_years"])
    else:
        tw_err = comparison_df[f"{method}_total_variance_err"]
        tw_valid = comparison_df[f"{method}_total_variance_valid"]
    comparison_summary.append({
        "method": method,
        "RMSE_to_mid": float(np.sqrt(np.mean(err ** 2))),
        "MAE_to_mid": float(np.mean(np.abs(err))),
        "spread_normalized_RMSE": float(np.sqrt(np.mean(comparison_df[spread_col] ** 2))),
        "mean_abs_spread_normalized_error": float(np.mean(np.abs(comparison_df[spread_col]))),
        "within_bid_ask_rate": float(np.mean(comparison_df[inside_col])),
        "total_variance_RMSE": float(np.sqrt(np.mean(np.asarray(tw_err)[np.asarray(tw_valid, dtype=bool)] ** 2))) if np.any(tw_valid) else np.nan,
    })
comparison_summary = pd.DataFrame(comparison_summary).sort_values("spread_normalized_RMSE")
display(comparison_summary)

fig = go.Figure()
for method in METHODS + ["bs"]:
    spread_col = "bs_spread_norm_err" if method == "bs" else f"{method}_spread_norm_err"
    by_time = comparison_df.groupby("quote_datetime")[spread_col].apply(lambda x: np.sqrt(np.mean(x ** 2))).reset_index(name="rmse")
    fig.add_trace(go.Scatter(x=by_time["quote_datetime"], y=by_time["rmse"], mode="lines+markers", name=method))
fig.update_layout(title="Heston methods vs fitted Black-Scholes: spread-normalized RMSE through time", xaxis_title="Quote time", yaxis_title="Spread-normalized RMSE", template="plotly_white", height=460)
fig.show()

fig = go.Figure()
for method in METHODS + ["bs"]:
    price_col = "bs_price" if method == "bs" else f"{method}_price"
    fig.add_trace(go.Scatter(x=comparison_df["mid"], y=comparison_df[price_col], mode="markers", name=method, opacity=0.55))
max_px = float(np.nanmax([comparison_df["mid"].max(), comparison_df[[f"{m}_price" for m in METHODS] + ["bs_price"]].max().max()]))
fig.add_trace(go.Scatter(x=[0, max_px], y=[0, max_px], mode="lines", name="perfect fit", line=dict(color="black", dash="dash")))
fig.update_layout(title="Market mids vs model prices", xaxis_title="Market mid", yaxis_title="Model price", template="plotly_white", height=620)
fig.show()


,method,RMSE_to_mid,MAE_to_mid,spread_normalized_RMSE,mean_abs_spread_normalized_error,within_bid_ask_rate,total_variance_RMSE
0,carr_madan,0.305915,0.198855,0.760558,0.583907,0.787500,0.000007
1,cos,0.303131,0.198130,0.762725,0.585997,0.789423,0.000007
3,bs,0.607746,0.412643,1.841939,1.329184,0.437500,0.000014
2,broadie_kaya,0.853427,0.650828,3.303109,2.477154,0.283654,0.000017


In [7]:
# Cell 7: full-scope Black-Scholes Greeks from market implied volatility.
# Output is a one-day interactive snapshot dashboard that plays through intraday quote times.
from plotly.subplots import make_subplots

if "comparison_df" not in globals():
    raise RuntimeError("Run the Black-Scholes comparison cell first.")

greek_input = comparison_df.copy()
greek_input["sigma_for_greeks"] = greek_input["market_iv"].where(np.isfinite(greek_input["market_iv"]), greek_input["bs_sigma_fit"])
greek_input = greek_input[np.isfinite(greek_input["sigma_for_greeks"]) & (greek_input["sigma_for_greeks"] > 0)].copy()


def bs_price_scalar(S, K, tau, r, sigma):
    return float(bs_call_price(S, np.array([K], dtype=float), tau, sigma, r)[0])


def greeks_fd_bs(S, K, tau, r, sigma):
    dS = max(0.0025 * S, 0.25)
    dV = max(1e-4, 0.0025 * sigma)
    dT = min(max(1.0 / 365.25 / 6.5, 1e-6), max(tau * 0.45, 1e-6))
    dr = 1e-4

    def V(S_, tau_, sig_, r_=r):
        return bs_price_scalar(S_, K, max(tau_, 1e-10), r_, max(sig_, 1e-8))

    V0 = V(S, tau, sigma)
    Vp = V(S + dS, tau, sigma)
    Vm = V(max(S - dS, 1e-8), tau, sigma)
    delta = (Vp - Vm) / (2 * dS)
    gamma = (Vp - 2 * V0 + Vm) / (dS ** 2)

    Vv_p = V(S, tau, sigma + dV)
    Vv_m = V(S, tau, max(sigma - dV, 1e-8))
    vega = (Vv_p - Vv_m) / (2 * dV)
    vomma = (Vv_p - 2 * V0 + Vv_m) / (dV ** 2)
    Vv_pp = V(S, tau, sigma + 2 * dV)
    Vv_mm = V(S, tau, max(sigma - 2 * dV, 1e-8))
    ultima = (Vv_pp - 3 * Vv_p + 3 * Vv_m - Vv_mm) / (2 * dV ** 3)

    tau_p = tau + dT
    tau_m = tau - dT
    theta = -(V(S, tau_p, sigma) - V0) / dT if tau_m <= 1e-10 else -(V(S, tau_p, sigma) - V(S, tau_m, sigma)) / (2 * dT)
    rho_rate = (V(S, tau, sigma, r + dr) - V(S, tau, sigma, r - dr)) / (2 * dr)

    V_sp = V(S + dS, tau, sigma + dV)
    V_sm = V(S + dS, tau, max(sigma - dV, 1e-8))
    V_mp = V(max(S - dS, 1e-8), tau, sigma + dV)
    V_mm = V(max(S - dS, 1e-8), tau, max(sigma - dV, 1e-8))
    vanna = (V_sp - V_sm - V_mp + V_mm) / (4 * dS * dV)

    def gamma_at(sig_, tau_):
        vv0 = V(S, tau_, sig_)
        vvp = V(S + dS, tau_, sig_)
        vvm = V(max(S - dS, 1e-8), tau_, sig_)
        return (vvp - 2 * vv0 + vvm) / (dS ** 2)

    gamma_p = gamma_at(sigma + dV, tau)
    gamma_m = gamma_at(max(sigma - dV, 1e-8), tau)
    zomma = (gamma_p - gamma_m) / (2 * dV)
    V2p = V(S + 2 * dS, tau, sigma)
    V2m = V(max(S - 2 * dS, 1e-8), tau, sigma)
    speed = (V2p - 3 * Vp + 3 * Vm - V2m) / (2 * dS ** 3)

    def delta_at_tau(tau_):
        return (V(S + dS, tau_, sigma) - V(max(S - dS, 1e-8), tau_, sigma)) / (2 * dS)

    def gamma_at_tau(tau_):
        return gamma_at(sigma, tau_)

    if tau_m <= 1e-10:
        charm = -(delta_at_tau(tau_p) - delta) / dT
        colour = -(gamma_at_tau(tau_p) - gamma) / dT
    else:
        charm = -(delta_at_tau(tau_p) - delta_at_tau(tau_m)) / (2 * dT)
        colour = -(gamma_at_tau(tau_p) - gamma_at_tau(tau_m)) / (2 * dT)

    return {
        "delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho_rate": rho_rate,
        "vanna": vanna, "vomma": vomma, "speed": speed, "charm": charm, "colour": colour,
        "zomma": zomma, "ultima": ultima,
    }


greek_rows = []
for row in greek_input.itertuples(index=False):
    vals = greeks_fd_bs(row.spot, row.strike, row.tau_years, RISK_FREE_RATE, row.sigma_for_greeks)
    out = {
        "quote_datetime": row.quote_datetime, "date": row.date, "time": row.time,
        "spot": row.spot, "strike": row.strike, "moneyness": row.moneyness, "mid": row.mid,
        "tau_years": row.tau_years, "tau_minutes": row.tau_minutes,
        "market_iv": row.market_iv, "sigma_for_greeks": row.sigma_for_greeks,
    }
    out.update(vals)
    greek_rows.append(out)

greeks_df = pd.DataFrame(greek_rows)
display(greeks_df.head())

GREEK_COLS = ["delta", "gamma", "vega", "theta", "rho_rate", "vanna", "vomma", "speed", "charm", "colour", "zomma", "ultima"]


def greek_surface_grid(day_df, greek):
    pivot = (
        day_df.pivot_table(index="quote_datetime", columns="strike", values=greek, aggfunc="mean")
        .sort_index()
        .sort_index(axis=1)
    )
    return pivot



def option_side_region_traces(day_df, pivot, greek, visible=True):
    spot_by_time = day_df.groupby("quote_datetime")["spot"].median().reindex(pivot.index)
    strikes = pivot.columns.astype(float).to_numpy()
    strike_min = float(np.nanmin(strikes))
    strike_max = float(np.nanmax(strikes))

    z_values = pivot.to_numpy(dtype=float)
    finite_z = z_values[np.isfinite(z_values)]
    if finite_z.size == 0:
        z_min, z_max = -1.0, 1.0
    else:
        z_min, z_max = float(finite_z.min()), float(finite_z.max())
        if np.isclose(z_min, z_max):
            pad = max(abs(z_min) * 0.10, 1e-8)
            z_min -= pad
            z_max += pad
    z_span = max(z_max - z_min, 1e-8)
    z_floor = z_min - 0.04 * z_span

    y_labels = [pd.Timestamp(t).strftime("%H:%M") for t in pivot.index]
    y = np.column_stack([y_labels, y_labels])
    spot = spot_by_time.to_numpy(dtype=float)
    n = len(pivot.index)
    gradient = np.tile(np.linspace(0.0, 1.0, 2), (n, 1))

    put_x = np.column_stack([np.full(n, strike_min), spot])
    put_z = np.full((n, 2), z_floor, dtype=float)
    call_x = np.column_stack([spot, np.full(n, strike_max)])
    call_z = np.full((n, 2), z_floor, dtype=float)
    atm_x = np.column_stack([spot, spot])
    atm_z = np.column_stack([np.full(n, z_min), np.full(n, z_max)])

    return [
        go.Surface(
            x=put_x, y=y, z=put_z,
            surfacecolor=gradient,
            colorscale=[[0.0, "rgb(0,20,255)"], [1.0, "rgb(0,255,255)"]],
            opacity=0.36,
            showscale=False,
            visible=visible,
            name="Synthetic put side (K < spot)",
            hovertemplate="synthetic put side<br>K=%{x:.0f}<br>time=%{y}<extra></extra>",
        ),
        go.Surface(
            x=call_x, y=y, z=call_z,
            surfacecolor=gradient,
            colorscale=[[0.0, "rgb(255,20,20)"], [1.0, "rgb(255,235,0)"]],
            opacity=0.36,
            showscale=False,
            visible=visible,
            name="Call side (K > spot)",
            hovertemplate="call side<br>K=%{x:.0f}<br>time=%{y}<extra></extra>",
        ),
        go.Surface(
            x=atm_x, y=y, z=atm_z,
            surfacecolor=np.zeros_like(atm_z, dtype=float),
            colorscale=[[0.0, "rgb(0,0,0)"], [1.0, "rgb(0,0,0)"]],
            opacity=0.45,
            showscale=False,
            visible=visible,
            name="Underlying / ATM divider",
            hovertemplate="spot=%{x:.2f}<br>time=%{y}<extra>Underlying divider</extra>",
        ),
    ]

def plot_greek_3d_field(greek, day=None):
    day = day or greeks_df["date"].min()
    day_df = greeks_df[greeks_df["date"] == day].sort_values(["quote_datetime", "strike"]).copy()
    if day_df.empty:
        raise ValueError(f"No Greek rows for day={day}")

    pivot = greek_surface_grid(day_df, greek)
    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=pivot.columns.astype(float),
        y=[pd.Timestamp(t).strftime("%H:%M") for t in pivot.index],
        z=pivot.to_numpy(dtype=float),
        colorscale="RdBu",
        colorbar=dict(title=greek),
        name=greek,
        hovertemplate="K=%{x:.0f}<br>time=%{y}<br>" + greek + "=%{z:.6g}<extra></extra>",
    ))
    for trace in option_side_region_traces(day_df, pivot, greek, visible=True):
        fig.add_trace(trace)
    fig.update_layout(
        title=f"{greek} 3D field - {day}",
        scene=dict(
            xaxis_title="Strike",
            yaxis_title="Quote time",
            zaxis_title=greek,
            camera=dict(eye=dict(x=1.45, y=1.55, z=0.9)),
        ),
        template="plotly_white",
        height=760,
    )
    fig.show()


def plot_all_greek_3d_fields(day=None, greek_cols=GREEK_COLS):
    day = day or greeks_df["date"].min()
    for greek in greek_cols:
        plot_greek_3d_field(greek, day=day)


def plot_greek_3d_dropdown(day=None, greek_cols=GREEK_COLS):
    day = day or greeks_df["date"].min()
    day_df = greeks_df[greeks_df["date"] == day].sort_values(["quote_datetime", "strike"]).copy()
    if day_df.empty:
        raise ValueError(f"No Greek rows for day={day}")

    fig = go.Figure()
    for idx, greek in enumerate(greek_cols):
        pivot = greek_surface_grid(day_df, greek)
        fig.add_trace(go.Surface(
            x=pivot.columns.astype(float),
            y=[pd.Timestamp(t).strftime("%H:%M") for t in pivot.index],
            z=pivot.to_numpy(dtype=float),
            colorscale="RdBu",
            colorbar=dict(title=greek),
            visible=(idx == 0),
            name=greek,
            hovertemplate="K=%{x:.0f}<br>time=%{y}<br>" + greek + "=%{z:.6g}<extra></extra>",
        ))
        for trace in option_side_region_traces(day_df, pivot, greek, visible=(idx == 0)):
            fig.add_trace(trace)

    buttons = []
    for idx, greek in enumerate(greek_cols):
        visible = [False] * (4 * len(greek_cols))
        base = 4 * idx
        visible[base] = True
        visible[base + 1] = True
        visible[base + 2] = True
        visible[base + 3] = True
        buttons.append({
            "label": greek,
            "method": "update",
            "args": [
                {"visible": visible},
                {
                    "title": f"{greek} 3D field - {day}",
                    "scene": {
                        "xaxis": {"title": "Strike"},
                        "yaxis": {"title": "Quote time"},
                        "zaxis": {"title": greek},
                        "camera": {"eye": {"x": 1.45, "y": 1.55, "z": 0.9}},
                    },
                },
            ],
        })

    fig.update_layout(
        title=f"{greek_cols[0]} 3D field - {day}",
        scene=dict(
            xaxis_title="Strike",
            yaxis_title="Quote time",
            zaxis_title=greek_cols[0],
            camera=dict(eye=dict(x=1.45, y=1.55, z=0.9)),
        ),
        updatemenus=[{
            "type": "dropdown",
            "x": 0.02,
            "y": 1.08,
            "buttons": buttons,
        }],
        template="plotly_white",
        height=780,
    )
    fig.show()


GREEK_FIELD_DAY = greeks_df["date"].min()
plot_greek_3d_dropdown(GREEK_FIELD_DAY)

# Uncomment for separate full-size figures for every Greek:
# plot_all_greek_3d_fields(GREEK_FIELD_DAY)


,quote_datetime,date,time,spot,strike,moneyness,mid,tau_years,tau_minutes,market_iv,...,vega,theta,rho_rate,vanna,vomma,speed,charm,colour,zomma,ultima
0,2022-05-16 09:31:00-04:00,2022-05-16,09:31,4010.9199,3965.0,0.988551,54.20,0.00074,389.0,0.572148,...,32.901802,-12705.725628,2.252145,-0.381917,31.489156,-0.007733,161.162537,1.267808,-0.003823,-1.608148e+07
1,2022-05-16 09:31:00-04:00,2022-05-16,09:31,4010.9199,3970.0,0.989798,50.35,0.00074,389.0,0.570050,...,34.785944,-13454.719834,2.182634,-0.361589,26.695688,-0.007489,153.544481,1.839824,-0.005052,-1.712776e+07
2,2022-05-16 09:31:00-04:00,2022-05-16,09:31,4010.9199,3975.0,0.991044,46.65,0.00074,389.0,0.568689,...,36.577879,-14184.699032,2.107891,-0.334431,21.757448,-0.007226,142.993409,2.422183,-0.006270,-1.809633e+07
3,2022-05-16 09:31:00-04:00,2022-05-16,09:31,4010.9199,3980.0,0.992291,43.00,0.00074,389.0,0.565161,...,38.186382,-14782.347613,2.031057,-0.303327,17.125527,-0.006956,129.948130,2.989211,-0.007469,-1.912874e+07
4,2022-05-16 09:31:00-04:00,2022-05-16,09:31,4010.9199,3985.0,0.993538,39.45,0.00074,389.0,0.560797,...,39.626969,-15282.082130,1.950630,-0.266935,12.765372,-0.006673,114.301373,3.534031,-0.008633,-2.016051e+07
